# Video Game Industry Analysis — What Makes a Best-Seller?
**Data Analyst Portfolio Project**  
Hugo Apolinário · 2025

**Business question:** What drives video game sales globally — and what can publishers learn from the data?

This notebook uses SQL queries (via SQLite) on a dataset of 16,000+ games to uncover what platforms, genres, publishers and release patterns drive global sales — translating findings into actionable industry recommendations.

---
## 0. Setup — install and import libraries

In [ ]:
import micropip
await micropip.install('seaborn')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sqlite3
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# sqlite3 is built into Python — no installation needed
# We create an in-memory SQL database and load our data into it
# This lets us write real SQL queries on our dataset
conn = sqlite3.connect(':memory:')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

print('Libraries loaded successfully!')

---
## 1. Load the data

In [ ]:
# Load the CSV file
df = pd.read_csv('vgsales.csv')

# Load the dataframe into our SQL database as a table called 'games'
# Now we can query it with real SQL
df.to_sql('games', conn, index=False, if_exists='replace')

print(f'Dataset loaded: {df.shape[0]:,} rows and {df.shape[1]} columns')
print('SQL table "games" is ready to query!')
df.head()

---
## 2. Data cleaning

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal rows before cleaning: {len(df):,}')

In [ ]:
# Drop rows with missing Year or Publisher
df.dropna(subset=['Year', 'Publisher'], inplace=True)

# Convert Year to integer
df['Year'] = df['Year'].astype(int)

# Remove years that are clearly wrong (future dates)
df = df[df['Year'] <= 2016]

# Reset index
df.reset_index(drop=True, inplace=True)

# Reload the cleaned data into SQL
df.to_sql('games', conn, index=False, if_exists='replace')

print(f'Rows after cleaning: {len(df):,}')
print(f'Year range: {df["Year"].min()} - {df["Year"].max()}')
print(f'Platforms: {df["Platform"].nunique()}')
print(f'Publishers: {df["Publisher"].nunique():,}')
print(f'Genres: {df["Genre"].nunique()}')
print('Data cleaning complete!')

---
## 3. SQL Analysis
### Query 1 — Which platforms generated the most global sales?

In [ ]:
q1 = pd.read_sql("""
    SELECT 
        Platform,
        COUNT(*) AS total_games,
        ROUND(SUM(Global_Sales), 2) AS total_global_sales_m,
        ROUND(AVG(Global_Sales), 2) AS avg_sales_per_game
    FROM games
    GROUP BY Platform
    ORDER BY total_global_sales_m DESC
    LIMIT 10
""", conn)

print('Top 10 platforms by global sales (millions USD):')
print(q1.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    q1['Platform'][::-1],
    q1['total_global_sales_m'][::-1],
    color=sns.color_palette('Blues_r', 10)
)
for bar in bars:
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.0f}M', va='center', fontsize=9)
ax.set_title('Top 10 platforms by global sales', fontsize=14, pad=15)
ax.set_xlabel('Total global sales (millions USD)')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.tight_layout()
plt.savefig('chart_01_platform_sales.png')
plt.show()
print('Chart saved!')

### Query 2 — Which genres are most profitable globally?

In [ ]:
q2 = pd.read_sql("""
    SELECT 
        Genre,
        COUNT(*) AS total_games,
        ROUND(SUM(Global_Sales), 2) AS total_global_sales_m,
        ROUND(SUM(NA_Sales), 2) AS na_sales_m,
        ROUND(SUM(EU_Sales), 2) AS eu_sales_m,
        ROUND(SUM(JP_Sales), 2) AS jp_sales_m,
        ROUND(AVG(Global_Sales), 3) AS avg_sales_per_game
    FROM games
    GROUP BY Genre
    ORDER BY total_global_sales_m DESC
""", conn)

print('Sales by genre (millions USD):')
print(q2.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    q2['Genre'][::-1],
    q2['total_global_sales_m'][::-1],
    color=sns.color_palette('Reds_r', len(q2))
)
for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.0f}M', va='center', fontsize=9)
ax.set_title('Global sales by genre', fontsize=14, pad=15)
ax.set_xlabel('Total global sales (millions USD)')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.tight_layout()
plt.savefig('chart_02_genre_sales.png')
plt.show()
print('Chart saved!')

### Query 3 — Which publishers dominate the market?

In [ ]:
q3 = pd.read_sql("""
    SELECT 
        Publisher,
        COUNT(*) AS total_games,
        ROUND(SUM(Global_Sales), 2) AS total_global_sales_m,
        ROUND(AVG(Global_Sales), 3) AS avg_sales_per_game,
        ROUND(MAX(Global_Sales), 2) AS best_selling_game_sales
    FROM games
    GROUP BY Publisher
    ORDER BY total_global_sales_m DESC
    LIMIT 10
""", conn)

print('Top 10 publishers by global sales:')
print(q3.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    q3['Publisher'][::-1],
    q3['total_global_sales_m'][::-1],
    color=sns.color_palette('Purples_r', 10)
)
for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.0f}M', va='center', fontsize=9)
ax.set_title('Top 10 publishers by global sales', fontsize=14, pad=15)
ax.set_xlabel('Total global sales (millions USD)')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.tight_layout()
plt.savefig('chart_03_publisher_sales.png')
plt.show()
print('Chart saved!')

### Query 4 — How has the industry grown over time?

In [ ]:
q4 = pd.read_sql("""
    SELECT 
        Year,
        COUNT(*) AS games_released,
        ROUND(SUM(Global_Sales), 2) AS total_global_sales_m,
        ROUND(AVG(Global_Sales), 3) AS avg_sales_per_game
    FROM games
    WHERE Year >= 1990
    GROUP BY Year
    ORDER BY Year
""", conn)

print('Industry growth by year:')
print(q4.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(13, 6))
color1 = '#2E86AB'
color2 = '#E50914'

ax1.bar(q4['Year'], q4['games_released'], color=color1, alpha=0.7, label='Games released')
ax1.set_xlabel('Year')
ax1.set_ylabel('Games released', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(q4['Year'], q4['total_global_sales_m'], color=color2,
         linewidth=2.5, marker='o', markersize=4, label='Global sales')
ax2.set_ylabel('Global sales (millions USD)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Video game industry growth over time', fontsize=14, pad=15)
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.xticks(q4['Year'][::2], rotation=45)
plt.tight_layout()
plt.savefig('chart_04_industry_growth.png')
plt.show()
print('Chart saved!')

### Query 5 — Are North American sales a reliable predictor of global success?

In [ ]:
q5 = pd.read_sql("""
    SELECT 
        Name,
        Platform,
        Genre,
        Publisher,
        NA_Sales,
        EU_Sales,
        JP_Sales,
        Global_Sales,
        ROUND(NA_Sales * 100.0 / Global_Sales, 1) AS na_pct_of_global
    FROM games
    WHERE Global_Sales > 1
    ORDER BY Global_Sales DESC
    LIMIT 100
""", conn)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    q5['NA_Sales'],
    q5['Global_Sales'],
    alpha=0.6,
    c=q5['EU_Sales'],
    cmap='RdYlGn',
    s=80,
    edgecolors='white',
    linewidth=0.5
)
plt.colorbar(scatter, label='EU Sales (millions)')

# Trend line
z = np.polyfit(q5['NA_Sales'], q5['Global_Sales'], 1)
p = np.poly1d(z)
ax.plot(sorted(q5['NA_Sales']), p(sorted(q5['NA_Sales'])),
        'r--', linewidth=1.5, alpha=0.8, label='Trend line')

ax.set_title('North American sales vs global sales\n(top 100 best-selling games)', fontsize=14, pad=15)
ax.set_xlabel('North America sales (millions USD)')
ax.set_ylabel('Global sales (millions USD)')
ax.legend()
plt.tight_layout()
plt.savefig('chart_05_na_vs_global.png')
plt.show()

corr = q5['NA_Sales'].corr(q5['Global_Sales'])
print(f'Correlation between NA sales and Global sales: {corr:.3f}')
print('Chart saved!')

### Query 6 — What is the best genre per region?

In [ ]:
q6 = pd.read_sql("""
    SELECT 
        Genre,
        ROUND(SUM(NA_Sales), 2) AS na_sales_m,
        ROUND(SUM(EU_Sales), 2) AS eu_sales_m,
        ROUND(SUM(JP_Sales), 2) AS jp_sales_m,
        ROUND(SUM(Other_Sales), 2) AS other_sales_m
    FROM games
    GROUP BY Genre
    ORDER BY na_sales_m DESC
""", conn)

print('Sales by genre and region (millions USD):')
print(q6.to_string(index=False))

x = range(len(q6))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar([i - width*1.5 for i in x], q6['na_sales_m'],   width, label='North America', color='#2E86AB')
ax.bar([i - width*0.5 for i in x], q6['eu_sales_m'],   width, label='Europe',        color='#1D9E75')
ax.bar([i + width*0.5 for i in x], q6['jp_sales_m'],   width, label='Japan',         color='#E50914')
ax.bar([i + width*1.5 for i in x], q6['other_sales_m'],width, label='Other',         color='#F4A261')

ax.set_title('Sales by genre and region', fontsize=14, pad=15)
ax.set_xlabel('Genre')
ax.set_ylabel('Total sales (millions USD)')
ax.set_xticks(list(x))
ax.set_xticklabels(q6['Genre'], rotation=30, ha='right')
ax.legend()
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('chart_06_genre_by_region.png')
plt.show()
print('Chart saved!')

### Query 7 — Top 10 best-selling games of all time

In [ ]:
q7 = pd.read_sql("""
    SELECT 
        Name,
        Platform,
        Year,
        Genre,
        Publisher,
        ROUND(Global_Sales, 2) AS global_sales_m
    FROM games
    ORDER BY Global_Sales DESC
    LIMIT 10
""", conn)

print('Top 10 best-selling games of all time:')
print(q7.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 6))
labels = [f"{row['Name']} ({row['Platform']})" for _, row in q7.iterrows()]
bars = ax.barh(
    labels[::-1],
    q7['global_sales_m'][::-1],
    color=sns.color_palette('Greens_r', 10)
)
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.1f}M', va='center', fontsize=9)
ax.set_title('Top 10 best-selling games of all time', fontsize=14, pad=15)
ax.set_xlabel('Global sales (millions USD)')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.tight_layout()
plt.savefig('chart_07_top_games.png')
plt.show()
print('Chart saved!')

---
## 4. Export cleaned data for Looker Studio dashboard

In [ ]:
# Export the cleaned dataset for Looker Studio
df.to_csv('vgsales_clean.csv', index=False)
print(f'Clean dataset exported: {len(df):,} rows')
print('Upload vgsales_clean.csv to Looker Studio to build your dashboard!')

---
## 5. Business recommendations

In [ ]:
top_platform  = q1.iloc[0]['Platform']
top_genre     = q2.iloc[0]['Genre']
top_publisher = q3.iloc[0]['Publisher']
peak_year     = int(q4.loc[q4['total_global_sales_m'].idxmax(), 'Year'])
na_corr       = round(q5['NA_Sales'].corr(q5['Global_Sales']), 3)

print('=' * 60)
print('  VIDEO GAME INDUSTRY — BUSINESS RECOMMENDATIONS')
print('=' * 60)
print(f'''
1. PRIORITISE {top_platform.upper()} FOR MAXIMUM REACH
   {top_platform} generated more global sales than any other
   platform. Publishers launching new titles should prioritise
   this platform for maximum commercial impact.

2. INVEST IN {top_genre.upper()} — THE HIGHEST-GROSSING GENRE
   {top_genre} games consistently outperform other genres
   globally. New publishers entering the market should
   consider this genre as their primary target.

3. NORTH AMERICA IS YOUR BELLWETHER MARKET
   NA sales correlate {na_corr} with global sales — extremely
   strong. A game that performs well in North America will
   almost certainly succeed globally. Prioritise NA launch
   strategy and marketing investment.

4. LEARN FROM {top_publisher.upper()}S PLAYBOOK
   {top_publisher} leads global sales by a significant margin.
   Their strategy — focusing on franchise titles, exclusive
   platform deals, and family-friendly content — is a proven
   formula worth studying.

5. THE INDUSTRY PEAKED IN {peak_year} — QUALITY OVER QUANTITY
   Global sales peaked in {peak_year} despite fewer releases
   than later years. The market rewards quality and franchise
   strength over volume. Publishers should focus on fewer,
   better games.
''')
print('=' * 60)

---
## 6. Summary statistics

In [ ]:
total_sales = df['Global_Sales'].sum()
best_game   = df.loc[df['Global_Sales'].idxmax(), 'Name']
best_sales  = df['Global_Sales'].max()

print('DATASET SUMMARY')
print(f'  Total games analysed  : {len(df):,}')
print(f'  Total global sales    : ${total_sales:,.0f} million')
print(f'  Year range            : {df["Year"].min()} - {df["Year"].max()}')
print(f'  Platforms covered     : {df["Platform"].nunique()}')
print(f'  Publishers covered    : {df["Publisher"].nunique():,}')
print(f'  Genres covered        : {df["Genre"].nunique()}')
print(f'  Best-selling game     : {best_game} (${best_sales:.1f}M)')
print(f'  Top platform          : {top_platform}')
print(f'  Top genre             : {top_genre}')
print(f'  Top publisher         : {top_publisher}')
print(f'  NA-Global correlation : {na_corr}')